<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/Phase9_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Phase 9 Evaluation Notebook (Updated)

## Overview
This notebook evaluates the **Phase 9** checkpoints (Best Baseline) on the test set. 
It performs:
- **Robust Checkpoint Discovery** (scans multiple paths)
- Metric calculation (F1, AUC, Accuracy)
- Confusion Matrix generation
- ROC Curve plotting

## Configuration
- **Checkpoints**: Scans `/content/drive/MyDrive/DAIC-WOZ_Datasets/` recursively
- **Dataset**: Full H5 OmniFusion Dataset
- **Tier**: Medium

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone/Update Repository
!git clone https://github.com/nithin12342/phase2.git /content/phase2 2>/dev/null || echo 'Repo exists'
%cd /content/phase2
!git pull origin main

# Install Dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers h5py pandas scikit-learn tqdm matplotlib seaborn

In [ ]:
import sys
import os
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, classification_report

# Add repo to path
sys.path.append('/content/phase2/ml_pipeline/h5_omnifusion')

from config.model_config import H5Config, ComputeTier
from src.models.h5_omnifusion import H5OmniFusion
from src.data.h5_dataset import create_h5_dataloaders_kfold

# === CONFIGURATION ===
TIER = "medium"
BATCH_SIZE = 16
MAX_SEQ_LEN = 256
DATA_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
CHECKPOINT_DIR = f"{DATA_ROOT}/checkpoints_phase9"  # Default
H5_DIR = f"{DATA_ROOT}/H5_OmniFusion_Output"
LABELS_PATH = f"{H5_DIR}/all_labels.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def load_checkpoint(path, device):
    try:
        # Fix for PyTorch 2.6+ default security restrictive loading
        # weights_only=False needed because we have numpy arrays in checkpoints
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                return checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                return checkpoint['state_dict']
            return checkpoint
        return checkpoint
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

# Robust Checkpoint Discovery
def find_checkpoints():
    print("🔍 Searching for checkpoints...")
    
    # 1. Check configured directory
    if os.path.exists(CHECKPOINT_DIR):
        ckpts = list(Path(CHECKPOINT_DIR).glob("*_best.pt"))
        if ckpts:
            print(f"   Found in default path: {CHECKPOINT_DIR}")
            return ckpts
            
    # 2. Check alternative common locations
    alternatives = [
        f"{DATA_ROOT}/checkpoints",
        f"{DATA_ROOT}/H5_OmniFusion_Output/checkpoints_phase9",
        f"{DATA_ROOT}/Phase9_Checkpoints",
        f"/content/phase2/checkpoints"
    ]
    
    for alt in alternatives:
        if os.path.exists(alt):
            ckpts = list(Path(alt).glob("*_best.pt"))
            if ckpts:
                print(f"✅ Found checkpoints in alternative path: {alt}")
                return ckpts
                
    # 3. Recursive search in DATA_ROOT (last resort)
    print("⚠️ No checkpoints found in standard paths. Searching recursively in DATA_ROOT (this may take a moment)...")
    found = list(Path(DATA_ROOT).rglob("*_best.pt"))
    if found:
         print(f"✅ Found recursively: {found[0].parent}")
    return found

checkpoints = find_checkpoints()

if not checkpoints:
    print(f"❌ CRITICAL: No checkpoints found anywhere in {DATA_ROOT}")
    print("Please check your Google Drive to locate where 'h5_omnifusion_medium_fold*_best.pt' files were saved.")
    best_ckpt = None
else:
    # Sort by modification time to get latest
    checkpoints.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    best_ckpt = checkpoints[0]
    print(f"✅ Found {len(checkpoints)} checkpoints.")
    print(f"👉 Using latest/best: {best_ckpt.name}")
    print(f"   (Location: {best_ckpt.parent})")

# Setup Model
config = H5Config.from_tier(ComputeTier(TIER))
model = H5OmniFusion(config)

if best_ckpt:
    state_dict = load_checkpoint(best_ckpt, device)
    if state_dict:
        model.load_state_dict(state_dict, strict=False)
        print("✅ Model weights loaded")
    else:
        print("❌ Failed to load weights")

model.to(device)
model.eval()
print("Model ready for evaluation")

In [ ]:
# Create Test Loader
# We use Fold 0 test split as representative
if best_ckpt:
    train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
        h5_dir=H5_DIR,
        labels_csv=LABELS_PATH,
        batch_size=BATCH_SIZE,
        fold_idx=0,
        n_folds=5,
        max_seq_len=MAX_SEQ_LEN
    )
    print(f"Test Set Size: {len(test_loader.dataset)}")
else:
    print("Skipping dataloader creation (no checkpoint)")

In [ ]:
# Recursive device mover
def to_device(data, device):
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: to_device(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [to_device(v, device) for v in data]
    return data

# Run Evaluation
all_preds = []
all_targets = []
all_probs = []

if best_ckpt:
    print("Running evaluation...")
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            try:
                # Recursive move to device
                inputs = {k: to_device(v, device) for k, v in batch.items() if k not in ['label', 'labels', 'target', 'targets']}
                
                # Handle different label keys
                if 'label' in batch:
                    targets = to_device(batch['label']['binary'], device)
                elif 'labels' in batch:
                    targets = to_device(batch['labels']['binary'], device)
                elif 'targets' in batch:
                    targets = to_device(batch['targets']['binary'], device)
                elif 'target' in batch:
                    targets = to_device(batch['target'], device)
                else:
                    # Inspect keys if label not found
                    print(f"⚠️ Batch {batch_idx}: Could not find label key. Available keys: {list(batch.keys())}")
                    continue
                
                # Forward
                outputs = model(inputs)
                probs = outputs[0]['binary_prob']
                preds = (probs >= 0.5).long()
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
            except Exception as e:
                 print(f"⚠️ Error in batch {batch_idx}: {e}")
                 # Simplified debug print to avoid JSON syntax error
                 print("    Batch keys:", list(batch.keys()))

    print("Evaluation complete!")
else:
    print("⚠️ Skipping evaluation - no checkpoint loaded")

In [ ]:
if best_ckpt:
    # Calculate Metrics
    if not all_targets:
        print("❌ No predictions made. Check batch processing errors.")
    else:
        y_true = np.array(all_targets)
        y_pred = np.array(all_preds)
        y_prob = np.array(all_probs)

        print("="*40)
        print("📊 Phase 9 Evaluation Results")
        print("="*40)

        if len(set(y_true)) > 1:
            print(classification_report(y_true, y_pred, target_names=['Non-Depressed', 'Depressed']))

            try:
                auc = roc_auc_score(y_true, y_prob)
                print(f"AUC-ROC: {auc:.4f}")

                # Plot Confusion Matrix
                cm = confusion_matrix(y_true, y_pred)
                plt.figure(figsize=(6,5))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Dep', 'Dep'], yticklabels=['Non-Dep', 'Dep'])
                plt.title(f"Confusion Matrix (AUC={auc:.2f})")
                plt.ylabel('True')
                plt.xlabel('Predicted')
                plt.show()

                # Plot ROC Curve
                fpr, tpr, _ = roc_curve(y_true, y_prob)
                plt.figure(figsize=(6,5))
                plt.plot(fpr, tpr, label=f'Model (AUC={auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--')
                plt.xlabel('False Positive Rate')
                plt.ylabel('True Positive Rate')
                plt.title('ROC Curve')
                plt.legend()
                plt.show()
            except Exception as e:
                print(f"Error plotting ROC/AUC: {e}")
        else:
            print("⚠️ Not enough class variety in test set for ROC/AUC")
            print(f"True Labels: {set(y_true)}")